In [1]:
import pandas as pd

df = pd.read_csv('returns_sustainability_dataset.csv')
df.head()

,Order_ID,Product_ID,User_ID,Order_Date,Product_Category,Product_Price,Order_Quantity,Discount_Applied,Shipping_Method,Payment_Method,...,Return_Status,Return_Reason,Days_to_Return,Order_Value,Return_Cost,Profit_Loss,CO2_Emissions,Packaging_Waste,CO2_Saved,Waste_Avoided
0,ORD00000,PROD0169,USER0195,2022-01-14,Clothing,1720.71,2,30.46,Next-Day,Wallet,...,Not Returned,No Return,0,2393.163468,0,2393.163468,2.0,0.4,2.0,0.4
1,ORD00001,PROD0318,USER1469,2022-01-03,Toys,744.06,5,29.62,Next-Day,Wallet,...,Returned,Size Issue,12,2618.347140,200,2418.347140,2.0,1.0,0.0,0.0
2,ORD00002,PROD0427,USER1812,2025-03-16,Clothing,983.68,5,47.80,Express,Wallet,...,Not Returned,No Return,0,2567.404800,0,2567.404800,1.5,1.0,1.5,1.0
3,ORD00003,PROD0323,USER1274,2024-11-06,Books,1855.65,2,2.90,Express,COD,...,Not Returned,No Return,0,3603.672300,0,3603.672300,1.5,0.4,1.5,0.4
4,ORD00004,PROD0325,USER0551,2023-06-07,Home Appliances,1770.97,5,44.42,Express,COD,...,Returned,Size Issue,11,4921.525630,200,4721.525630,1.5,1.0,0.0,0.0


In [11]:
print("Shape:", df.shape)
print()
print("Column names and data types:")
print(df.dtypes)

Shape: (5000, 23)

Column names and data types:
Order_ID             object
Product_ID           object
User_ID              object
Order_Date           object
Product_Category     object
Product_Price       float64
Order_Quantity        int64
Discount_Applied    float64
Shipping_Method      object
Payment_Method       object
User_Age              int64
User_Gender          object
User_Location        object
Return_Status        object
Return_Reason        object
Days_to_Return        int64
Order_Value         float64
Return_Cost           int64
Profit_Loss         float64
CO2_Emissions       float64
Packaging_Waste     float64
CO2_Saved           float64
Waste_Avoided       float64
dtype: object


In [14]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Total missing values:", df.isnull().sum().sum())
print()
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Order_IDs:", df['Order_ID'].duplicated().sum())

Missing values per column:
Order_ID            0
Product_ID          0
User_ID             0
Order_Date          0
Product_Category    0
Product_Price       0
Order_Quantity      0
Discount_Applied    0
Shipping_Method     0
Payment_Method      0
User_Age            0
User_Gender         0
User_Location       0
Return_Status       0
Return_Reason       0
Days_to_Return      0
Order_Value         0
Return_Cost         0
Profit_Loss         0
CO2_Emissions       0
Packaging_Waste     0
CO2_Saved           0
Waste_Avoided       0
dtype: int64

Total missing values: 0

Duplicate rows: 0
Duplicate Order_IDs: 0


In [15]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
print(df['Order_Date'].dtype)
print("Date range:", df['Order_Date'].min(), "to", df['Order_Date'].max())

datetime64[ns]
Date range: 2022-01-01 00:00:00 to 2025-09-03 00:00:00


In [16]:
df['Is_Returned'] = (df['Return_Status'] == 'Returned').astype(int)

df['Return_Speed_Bucket'] = pd.cut(
    df['Days_to_Return'],
    bins=[-1, 7, 21, 60],
    labels=['0-7 days (fast)', '8-21 days (medium)', '22+ days (slow)']
)

df['Discount_Bucket'] = pd.cut(
    df['Discount_Applied'],
    bins=[-1, 10, 20, 30, 40, 50],
    labels=['0-10%', '11-20%', '21-30%', '31-40%', '41-50%']
)

df[['Order_ID', 'Is_Returned', 'Days_to_Return', 'Return_Speed_Bucket', 'Discount_Applied', 'Discount_Bucket']].head()

,Order_ID,Is_Returned,Days_to_Return,Return_Speed_Bucket,Discount_Applied,Discount_Bucket
0,ORD00000,0,0,0-7 days (fast),30.46,31-40%
1,ORD00001,1,12,8-21 days (medium),29.62,21-30%
2,ORD00002,0,0,0-7 days (fast),47.80,41-50%
3,ORD00003,0,0,0-7 days (fast),2.90,0-10%
4,ORD00004,1,11,8-21 days (medium),44.42,41-50%


In [17]:
from scipy import stats

returned_only = df[df['Is_Returned'] == 1]

ct = pd.crosstab(returned_only['Return_Speed_Bucket'], returned_only['Return_Reason'], normalize='index') * 100
print("Return reason % WITHIN each speed bucket:")
print(ct.round(1))

chi2, p_value, dof, expected = stats.chi2_contingency(
    pd.crosstab(returned_only['Return_Speed_Bucket'], returned_only['Return_Reason'])
)
print()
print(f"Chi-square p-value: {p_value:.4f}")
if p_value < 0.05:
    print("=> Statistically significant relationship found.")
else:
    print("=> No statistically significant relationship. The 'fast vs slow' theory does NOT hold up.")

Return reason % WITHIN each speed bucket:
Return_Reason        Changed Mind  Defective  Size Issue  Wrong Item
Return_Speed_Bucket                                                 
0-7 days (fast)              19.0       25.3        27.8        27.8
8-21 days (medium)           28.5       27.1        23.8        20.7
22+ days (slow)              25.9       26.2        23.1        24.9

Chi-square p-value: 0.4714
=> No statistically significant relationship. The 'fast vs slow' theory does NOT hold up.


In [18]:
# Define a reusable function to compare a numeric column between returned vs kept orders
def compare_groups(column_name):
    # Get the values of this column, but only for returned orders
    returned_vals = df[df['Is_Returned'] == 1][column_name]
    
    # Get the values of this column, but only for orders that were NOT returned
    not_returned_vals = df[df['Is_Returned'] == 0][column_name]
    
    # Run a statistical test (t-test) comparing the two groups' averages
    t_stat, p_val = stats.ttest_ind(returned_vals, not_returned_vals)
    
    # Print the results in a readable format
    print(f"{column_name}:")
    print(f"  Returned orders average:     {returned_vals.mean():.2f}")
    print(f"  Not-returned orders average: {not_returned_vals.mean():.2f}")
    print(f"  p-value: {p_val:.4f}  {'=> SIGNIFICANT' if p_val < 0.05 else '=> not significant'}")
    print()

# Now actually call the function for each column we want to test
compare_groups('Product_Price')   # does price affect returns?
compare_groups('User_Age')        # does customer age affect returns?
compare_groups('Discount_Applied') # does discount level affect returns?

Product_Price:
  Returned orders average:     1062.32
  Not-returned orders average: 1051.02
  p-value: 0.5084  => not significant

User_Age:
  Returned orders average:     41.45
  Not-returned orders average: 41.59
  p-value: 0.7448  => not significant

Discount_Applied:
  Returned orders average:     26.80
  Not-returned orders average: 24.28
  p-value: 0.0000  => SIGNIFICANT



In [19]:
# Group orders by their discount bucket, and calculate the return rate within each group
discount_analysis = df.groupby('Discount_Bucket', observed=True)['Is_Returned'].agg(['mean', 'count'])

# 'mean' of a 0/1 column gives us a proportion -- multiply by 100 to turn it into a percentage
discount_analysis['return_rate_pct'] = (discount_analysis['mean'] * 100).round(2)

# Rename 'count' to something clearer
discount_analysis = discount_analysis.rename(columns={'count': 'total_orders'})

# Show just the columns we care about
print(discount_analysis[['total_orders', 'return_rate_pct']])

                 total_orders  return_rate_pct
Discount_Bucket                               
0-10%                     977            24.05
11-20%                   1005            26.87
21-30%                   1054            25.90
31-40%                    943            35.21
41-50%                   1021            33.30


In [20]:
# Step 1: Count how many returns each category+reason combination has
category_reason_counts = pd.crosstab(returned_only['Product_Category'], returned_only['Return_Reason'])

print("Raw counts:")
print(category_reason_counts)
print()

# Step 2: Turn those counts into percentages, so each category's row adds up to 100%
category_reason_pct = category_reason_counts.div(category_reason_counts.sum(axis=1), axis=0) * 100

print("As percentages (each row adds up to 100%):")
print(category_reason_pct.round(1))

Raw counts:
Return_Reason     Changed Mind  Defective  Size Issue  Wrong Item
Product_Category                                                 
Books                       47         41          44          40
Clothing                   166        179         151         165
Electronics                 93         77          74          70
Home Appliances             31         33          30          28
Toys                        42         52          42          45

As percentages (each row adds up to 100%):
Return_Reason     Changed Mind  Defective  Size Issue  Wrong Item
Product_Category                                                 
Books                     27.3       23.8        25.6        23.3
Clothing                  25.1       27.1        22.8        25.0
Electronics               29.6       24.5        23.6        22.3
Home Appliances           25.4       27.0        24.6        23.0
Toys                      23.2       28.7        23.2        24.9


In [23]:
# Round all numeric columns to 2 decimal places to clean up floating-point display noise
numeric_columns = df.select_dtypes(include='float64').columns
df[numeric_columns] = df[numeric_columns].round(2)
df[numeric_columns].head()

,Product_Price,Discount_Applied,Order_Value,Profit_Loss,CO2_Emissions,Packaging_Waste,CO2_Saved,Waste_Avoided
0,1720.71,30.46,2393.16,2393.16,2.0,0.4,2.0,0.4
1,744.06,29.62,2618.35,2418.35,2.0,1.0,0.0,0.0
2,983.68,47.80,2567.40,2567.40,1.5,1.0,1.5,1.0
3,1855.65,2.90,3603.67,3603.67,1.5,0.4,1.5,0.4
4,1770.97,44.42,4921.53,4721.53,1.5,1.0,0.0,0.0


In [24]:
# Saving the dataframe to a new CSV file
df.to_csv('returns_cleaned_for_tableau.csv', index=False)

print("Saved: returns_cleaned_for_tableau.csv")
print("Shape:", df.shape)
print("Columns:", list(df.columns))

Saved: returns_cleaned_for_tableau.csv
Shape: (5000, 26)
Columns: ['Order_ID', 'Product_ID', 'User_ID', 'Order_Date', 'Product_Category', 'Product_Price', 'Order_Quantity', 'Discount_Applied', 'Shipping_Method', 'Payment_Method', 'User_Age', 'User_Gender', 'User_Location', 'Return_Status', 'Return_Reason', 'Days_to_Return', 'Order_Value', 'Return_Cost', 'Profit_Loss', 'CO2_Emissions', 'Packaging_Waste', 'CO2_Saved', 'Waste_Avoided', 'Is_Returned', 'Return_Speed_Bucket', 'Discount_Bucket']
